In [ ]:
from langgraph.graph import StateGraph, START , END
from dotenv import load_dotenv
from typing import TypedDict
#from langchain_huggingface import ChatHuggingFace
from langchain_openai import ChatOpenAI

In [ ]:
load_dotenv()

In [ ]:
model = ChatOpenAI()

In [ ]:
class Blogstate(TypedDict):
    title = str
    outline = str
    content = str

In [ ]:
def create_outline(state: Blogstate)-> Blogstate:
    
    #fetch title
    title = state['title']
    
    # call llm gen outline
    prompt = f"Generate the detailed outline for a blog on the topic: {title} "
    outline = model.invoke(prompt).content
    
    #update the state of the outline
    outline = state['outline']    
    
    return state
    
def create_content(state: Blogstate)-> Blogstate:
    
    title = state['title']
    outline = state['outline']
    
    prompt = f'Make the content on the title :{title} havinfg the outline sa {outline}'
        
    content = model.invoke(prompt).content
    
    # update the state
    state['state'] = content
    
    return state

In [ ]:
graph = StateGraph(Blogstate)

# create the nodes
graph.node('outline',create_outline )
graph.node('content',create_content)

#create edges

graph.edges(START, 'outline')
graph.edges('outline','content')
graph.edges('content', END)

#compile
workflow = graph.compile()

In [ ]:
initial_state = {'title' : "Rise of AI in India"}

final_state = workflow.invoke(initial_state)

print(final_state)